In [ ]:
from utils import get_stage_packages
get_stage_packages()
! pip install -r pip-requirements.txt

Processing ./dist/january_ml-0.0.1-py3-none-any.whl (from -r project-requirements.txt (line 1))
january-ml is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [ ]:
from january_ml.utils import version_data

from sklearn.linear_model import LinearRegression

from snowflake.snowpark.session import Session
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.registry import Registry
from snowflake.ml.dataset import Dataset

/opt/anaconda3/envs/jan/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Create session
session = Session.builder.getOrCreate()

In [ ]:
# Get Data

fs = FeatureStore(
    session=session,
    database=session.get_current_database(),
    name="SHARED_WORK",
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

fv = fs.get_feature_view("EXAMPLE_FEATURES",version="1")
df = fs.read_feature_view(fv).sample(n=1000)

/opt/anaconda3/envs/jan/lib/python3.10/site-packages/snowflake/ml/feature_store/feature_store.py:303: UserWarning: The current snowflake-ml-python version out of date, package upgrade recommended (current=1.17.0, recommended>=1.19.0)
  self._check_feature_store_object_versions()


In [7]:
# Save as dataset
ds_name = "TRAIN_DATASET"
df = df.select("HOUR","OPENED")
ds = Dataset.create(session=session, name=ds_name, exist_ok=True)
version_name = version_data(df)
ds_version = ds.create_version(version=version_name, input_dataframe=df, label_cols=["OPENED"])

In [8]:
# Train Model
df = df.to_pandas()

X = df.drop(columns=["OPENED"])
y = df[["OPENED"]]

lr = LinearRegression()
lr.fit(X, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [11]:
# Register model
reg = Registry(session=session)

model_name = "MODEL_EX1"
mv = reg.log_model(
    model=lr, 
    model_name=model_name, 
    sample_input_data=ds_version.read.to_snowpark_dataframe().drop("OPENED").limit(100), 
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"] )

reg.show_models()


Logging model: creating model manifest...:  33%|███▎      | 2/6 [00:00<00:00,  4.81it/s]  

/opt/anaconda3/envs/jan/lib/python3.10/site-packages/snowflake/ml/registry/_manager/model_parameter_reconciler.py:72: UserWarning: `relax_version` is not set and therefore defaulted to True. Dependency version constraints relaxed from ==x.y.z to >=x.y, <(x+1). To use specific dependency versions for compatibility, reproducibility, etc., set `options={'relax_version': False}` when logging the model.
  reconciled_options = self._reconcile_relax_version(reconciled_options, reconciled_target_platforms)
/opt/anaconda3/envs/jan/lib/python3.10/site-packages/snowflake/ml/model/model_signature.py:71: UserWarning: The sample input has 100 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(


Model logged successfully.: 100%|██████████| 6/6 [01:12<00:00, 12.14s/it]                          


,created_on,name,model_type,database_name,schema_name,comment,owner,default_version_name,versions,aliases
0,2025-12-10 16:08:40.855000+00:00,MODEL_EX1,USER_MODEL,ML_COLLAB_DEV_DB,SHARED_WORK,None,EXTERNAL_SNOWFLAKE_ARCHITECTS,BROWN_ANT_4,"[""BROWN_ANT_4""]","{""DEFAULT"":""BROWN_ANT_4"",""FIRST"":""BROWN_ANT_4""..."


In [12]:
# promote to default
base_model = reg.get_model(model_name)
base_model.default = mv